In [1]:
import pandas as pd
import scipy.io as sio

#load the .mat file into a dictionary
#extract the variable B005 from the loaded .mat file
mat_db = sio.loadmat('dataset/BatteryAgingARC_49_50_51_52/B0051.mat') ['B0051']

In [2]:
def to_df(mat_db):
    """Returns one pd.DataFrame per cycle type"""
    #cycle type: charge, discharge, impedance

    # Features common for every cycle
    cycles_cols = ['type', 'ambient_temperature', 'time']

    # Features monitored during the cycle
    features_cols = {
        'discharge': ['Voltage_measured', 'Current_measured', 'Temperature_measured', 
                'Current_charge', 'Voltage_charge', 'Time'],
        'charge': ['Voltage_measured', 'Current_measured', 'Temperature_measured', 
                    'Current_charge', 'Voltage_charge', 'Time', 'Capacity'],
        'impedance': ['Sense_current', 'Battery_current', 'Current_ratio',
                    'Battery_impedance', 'Rectified_impedance', 'Re', 'Rct']
    }

    # Define one pd.DataFrame per cycle type
    #Create an empty Pandas DataFrame for each cycle type defined in features_cols
    df = {key: pd.DataFrame() for key in features_cols.keys()}

    # Get every cycle
    print(f'Number of cycles: {mat_db[0][0][0].shape[1]}')
    cycles = [[row.flat[0] for row in line] for line in mat_db[0][0][0][0]]

    # Get measures for every cycle
    for cycle_id, cycle_data in enumerate(cycles):
        tmp = pd.DataFrame() #store data for the current cycle

        # Data series for every cycle
        #Retrieve the data series for the current cycle from the last element of cycle_data
        features_x_cycle = cycle_data[-1]

        # Get features for the specific cycle type
        features = features_cols[cycle_data[0]]
        
        for feature, data in zip(features, features_x_cycle):
            if len(data[0]) > 1:
                # Correct number of records
                tmp[feature] = data[0]
            else:
                # Single value, so assign it to all rows
                tmp[feature] = data[0][0]
        
        # Add columns common to the cycle measurements
        tmp['id_cycle'] = cycle_id
        for k, col in enumerate(cycles_cols):
            tmp[col] = cycle_data[k]
        
        # Append cycle data to the right pd.DataFrame
        cycle_type = cycle_data[0]
        #The original used method was append, but it was deprecated 
        #It is repalced by pandas.concat
        df[cycle_type] = pd.concat([df[cycle_type], tmp], ignore_index=True)
    
    return df

In [3]:
dfs = to_df(mat_db)

dfs['discharge'].tail(10)

Number of cycles: 62


,Voltage_measured,Current_measured,Temperature_measured,Current_charge,Voltage_charge,Time,id_cycle,type,ambient_temperature,time
5924,3.723512,-0.003927,13.800770,0.0004,0.0,1610.547,58,discharge,4,2010.0
5925,3.725538,-0.004759,13.737293,0.0004,0.0,1621.454,58,discharge,4,2010.0
5926,3.727474,-0.004164,13.544136,0.0004,0.0,1632.391,58,discharge,4,2010.0
5927,3.729227,-0.004491,13.246243,0.0004,0.0,1643.297,58,discharge,4,2010.0
5928,3.731000,-0.004520,13.217434,0.0004,0.0,1654.282,58,discharge,4,2010.0
5929,3.732600,-0.003068,12.971872,0.0004,0.0,1665.172,58,discharge,4,2010.0
5930,3.734241,-0.004138,12.870139,0.0004,0.0,1676.110,58,discharge,4,2010.0
5931,3.735807,-0.005340,12.817266,0.0004,0.0,1686.985,58,discharge,4,2010.0
5932,3.737421,-0.002259,12.648285,0.0004,0.0,1697.844,58,discharge,4,2010.0
5933,3.738739,-0.004159,12.545432,0.0004,0.0,1708.860,58,discharge,4,2010.0


In [4]:
# Save each dataframe to a CSV file
dfs['discharge'].to_csv('discharge_data.csv', index=False)
dfs['charge'].to_csv('charge_data.csv', index=False)
dfs['impedance'].to_csv('impedance_data.csv', index=False)